# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/Alpeshmore/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 163, done.
remote: Counting objects: 100% (163/163), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 163 (delta 68), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (163/163), 1.95 MiB | 3.56 MiB/s, done.
Resolving deltas: 100% (68/68), done.


In [2]:
# ML-10 — Setup

from pathlib import Path
import numpy as np
import pandas as pd

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

# Try to reuse objects from ML-08/ML-09.
# If they do not exist, load the processed feature vector.

if "df" not in globals():

    possible_paths = [
        Path("data/processed/refresh_feature_vector.csv"),
        Path("data/raw/content_refresh_anonymized.csv"),
        Path("/content/flyrank-ai/data/processed/refresh_feature_vector.csv"),
        Path("/content/flyrank-ai/data/raw/content_refresh_anonymized.csv"),
        Path("/content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv"),
        Path("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"),
    ]

    data_path = next((p for p in possible_paths if p.exists()), None)

    if data_path is None:
        raise FileNotFoundError(
            "Dataset not found. Run the earlier ML notebooks first "
            "or place the FlyRank dataset in data/raw/."
        )

    df = pd.read_csv(data_path)
    print("Loaded:", data_path)

# Create target if necessary
if "is_declining_label" not in df.columns:
    df["is_declining_label"] = (
        df["trend_direction"]
        .astype(str)
        .str.lower()
        .eq("down")
        .astype(int)
    )

df["is_declining_label"] = df["is_declining_label"].astype(int)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Clients:", df["client_id"].nunique())

Loaded: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
Rows: 30000
Columns: 45
Clients: 32


## 1. Ranked actions + reason codes

The playbook ranks pages for **human review**, rather than automatically deciding that a page must be refreshed.

The ranking combines observed visibility, freshness risk, search-position opportunity, and content depth. Reason codes explain why a page was ranked highly in simple words.

The main action is **REVIEW**. A page should only move from review to an actual content change after a person checks the evidence.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# Build the ranked action queue
# ---------------------------------------------------------

queue = df.copy()

# Safe numeric conversions
queue["impressions_90d"] = pd.to_numeric(
    queue["impressions_90d"], errors="coerce"
).fillna(0)

queue["sessions_90d"] = pd.to_numeric(
    queue["sessions_90d"], errors="coerce"
).fillna(0)

queue["word_count"] = pd.to_numeric(
    queue["word_count"], errors="coerce"
).fillna(0)

queue["days_since_last_update"] = pd.to_numeric(
    queue["days_since_last_update"], errors="coerce"
).fillna(0)

queue["content_age_days"] = pd.to_numeric(
    queue["content_age_days"], errors="coerce"
).fillna(0)

queue["avg_position"] = pd.to_numeric(
    queue["avg_position"], errors="coerce"
).fillna(0)

queue["ctr"] = pd.to_numeric(
    queue["ctr"], errors="coerce"
).fillna(0)

queue["engagement_rate"] = pd.to_numeric(
    queue["engagement_rate"], errors="coerce"
).fillna(0)

queue["scroll_rate"] = pd.to_numeric(
    queue["scroll_rate"], errors="coerce"
).fillna(0)


# ---------------------------------------------------------
# Score components
# ---------------------------------------------------------

def percentile_score(series):
    """Convert a numeric signal to a 0-1 percentile score."""
    return series.rank(method="average", pct=True).fillna(0)


queue["visibility_score"] = percentile_score(
    np.log1p(queue["impressions_90d"])
)

queue["freshness_risk_score"] = percentile_score(
    queue["days_since_last_update"]
)


# Position opportunity:
# Lower average position number = stronger search position.
# 1-20 is the useful range for this review rule.

position_clipped = queue["avg_position"].clip(
    lower=1,
    upper=20
)

queue["position_opportunity_score"] = (
    ((21 - position_clipped) / 20)
    .where(queue["avg_position"] > 0, 0)
)


# Depth gap:
# Pages below 1200 words receive a larger depth-gap signal.

queue["depth_gap_score"] = (
    ((1200 - queue["word_count"]) / 1200)
    .clip(0, 1)
)


# ---------------------------------------------------------
# Transparent baseline score
# ---------------------------------------------------------

queue["action_score"] = (
    0.40 * queue["visibility_score"]
    + 0.30 * queue["freshness_risk_score"]
    + 0.25 * queue["position_opportunity_score"]
    + 0.05 * queue["depth_gap_score"]
)

queue["action_score"] = (
    queue["action_score"]
    .clip(0, 1)
    * 100
)


# ---------------------------------------------------------
# Reason codes
# ---------------------------------------------------------

def get_reason_codes(row):

    reasons = []

    if (
        row["days_since_last_update"] >= 180
        and row["impressions_90d"] >= 500
    ):
        reasons.append("stale_visible_page")

    if (
        row["word_count"] < 1200
        and row["impressions_90d"] >= 250
    ):
        reasons.append("thin_visible_page")

    if (
        0 < row["avg_position"] <= 10
        and row["content_age_days"] >= 180
    ):
        reasons.append("page_one_decay_risk")

    if (
        row["impressions_90d"] >= 500
        and row["avg_position"] > 0
        and row["avg_position"] <= 20
        and row["ctr"] < 0.50
    ):
        reasons.append("low_ctr_visible_page")

    if (
        row["sessions_90d"] >= 30
        and (
            row["engagement_rate"] < 30
            or row["scroll_rate"] < 30
        )
    ):
        reasons.append("low_engagement_visible_page")

    if not reasons:
        reasons.append("monitor")

    return "|".join(reasons)


queue["reason_codes"] = queue.apply(
    get_reason_codes,
    axis=1
)


# ---------------------------------------------------------
# Final action
# ---------------------------------------------------------

queue["action"] = np.where(
    queue["reason_codes"].eq("monitor"),
    "MONITOR",
    "REVIEW"
)


# Rank highest score first
queue = queue.sort_values(
    ["action_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)


print("Queue created.")
print("Rows:", len(queue))
print("\nAction counts:")
print(queue["action"].value_counts())

print("\nTop 20:")
print(
    queue[
        [
            "rank",
            "action_score",
            "action",
            "reason_codes",
            "impressions_90d",
            "avg_position",
            "word_count",
            "days_since_last_update"
        ]
    ].head(20).to_string(index=False)
)

Queue created.
Rows: 30000

Action counts:
action
REVIEW     18296
MONITOR    11704
Name: count, dtype: int64

Top 20:
 rank  action_score action                                                                           reason_codes  impressions_90d  avg_position  word_count  days_since_last_update
    1     94.031333 REVIEW                      thin_visible_page|page_one_decay_risk|low_engagement_visible_page           309192           2.0         0.0                     104
    2     93.610333 REVIEW                      thin_visible_page|page_one_decay_risk|low_engagement_visible_page            44437           1.7         0.0                     104
    3     93.384667 REVIEW                      thin_visible_page|page_one_decay_risk|low_engagement_visible_page            51233           2.0         0.0                     104
    4     93.208333 REVIEW                      thin_visible_page|page_one_decay_risk|low_engagement_visible_page            29717           1.5         0.0 

In [4]:
# Check that the queue is actually ranked correctly

assert queue["rank"].is_monotonic_increasing
assert queue["action_score"].is_monotonic_decreasing

print("Ranking check: PASS")

Ranking check: PASS


## 2. Intended use and limits

### Intended use

The playbook is intended for an SEO/content team to prioritize pages for human review. It is most useful when there are many pages and the team needs a consistent way to decide which observed signals deserve attention first.

The score is decision-support. It does not automatically mean that a page should be rewritten.

### Limits

The underlying label is based on observed decline, so the playbook does not prove that refreshing a page will improve its future performance.

The data is also observational. Traffic, rankings, content age, and engagement can move for many reasons that are not represented in the score.

The ranking should not be treated as equally reliable for every page, especially pages with very little traffic or missing search-position information.

The playbook should therefore be used as a prioritization layer followed by human review.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# Check the population being recommended for review
# ---------------------------------------------------------

review_queue = queue[
    queue["action"] == "REVIEW"
].copy()

monitor_queue = queue[
    queue["action"] == "MONITOR"
].copy()

print("REVIEW pages:", len(review_queue))
print("MONITOR pages:", len(monitor_queue))

print("\nReview percentage:")
print(
    round(
        len(review_queue) / len(queue) * 100,
        2
    ),
    "%"
)

print("\nMedian impressions:")
print(
    "Review:",
    round(review_queue["impressions_90d"].median(), 1)
)

print(
    "Monitor:",
    round(monitor_queue["impressions_90d"].median(), 1)
)

print("\nPages with no position data:")
print(
    round(
        (queue["avg_position"] == 0).mean() * 100,
        2
    ),
    "%"
)

REVIEW pages: 18296
MONITOR pages: 11704

Review percentage:
60.99 %

Median impressions:
Review: 2086.0
Monitor: 102.0

Pages with no position data:
4.02 %


## 3. Human review + the no-go list

Before acting on a recommendation, a person should check:

1. Whether the page still matches the search intent.
2. Whether the content is factually correct and current.
3. Whether the observed traffic and ranking signals are based on enough data to be useful.
4. Whether the page has business or editorial importance that the score does not capture.
5. Whether a recent site, algorithm, seasonality, or technical change could explain the observed movement.
6. Whether the proposed refresh has a clear purpose.

### No-go list

The system should never automatically:

* publish or rewrite content;
* delete a page;
* change a URL;
* make a medical, legal, financial, or other high-stakes claim;
* infer causality from the score;
* declare that a refresh will improve rankings;
* override an editorial or business decision;
* expose private client information.

The model recommends **where to look first**. A human decides **what to do**.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# Human-review checklist attached to the queue
# ---------------------------------------------------------

review_columns = [
    "rank",
    "action_score",
    "action",
    "reason_codes",
    "impressions_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "content_age_days",
    "days_since_last_update",
]

review_columns = [
    c for c in review_columns
    if c in queue.columns
]

top_review = queue[
    queue["action"] == "REVIEW"
][review_columns].head(20)

print("Top pages requiring human review:")
print(
    top_review.to_string(index=False)
)

print("\nHuman review gate: REQUIRED")
print("Automatic publishing: NOT ALLOWED")

Top pages requiring human review:
 rank  action_score action                                                                           reason_codes  impressions_90d  sessions_90d  avg_position  ctr  word_count  content_age_days  days_since_last_update
    1     94.031333 REVIEW                      thin_visible_page|page_one_decay_risk|low_engagement_visible_page           309192          1098           2.0 0.87         0.0               445                     104
    2     93.610333 REVIEW                      thin_visible_page|page_one_decay_risk|low_engagement_visible_page            44437           217           1.7 0.52         0.0               330                     104
    3     93.384667 REVIEW                      thin_visible_page|page_one_decay_risk|low_engagement_visible_page            51233           386           2.0 0.88         0.0               313                     104
    4     93.208333 REVIEW                      thin_visible_page|page_one_decay_risk|low_enga

## 4. Monitoring / retrain triggers

I would monitor the playbook using both model performance and data quality.

A retraining or review trigger would be:

* Precision@50 falls materially below the validated benchmark on a new labeled sample.
* The share of declining pages changes substantially from the validation period.
* Traffic or search-position distributions shift substantially.
* Important features develop much more missing data.
* New content types or workflows appear that were not represented in training.
* The reason-code distribution changes sharply.

I would not retrain only because time has passed. A meaningful change in the data or measured performance is a stronger reason to investigate.

As a practical starting point, I would investigate if Precision@50 falls by around **20% relative to the validated benchmark**, or if the target rate or major feature distributions show a large shift.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# Monitoring baseline statistics
# ---------------------------------------------------------

monitoring_stats = {
    "rows": len(queue),
    "clients": queue["client_id"].nunique(),
    "decline_rate": queue["is_declining_label"].mean(),
    "review_rate": (queue["action"] == "REVIEW").mean(),
    "median_impressions": queue["impressions_90d"].median(),
    "median_sessions": queue["sessions_90d"].median(),
    "median_word_count": queue["word_count"].median(),
    "median_days_since_update": queue["days_since_last_update"].median(),
    "no_position_rate": (queue["avg_position"] == 0).mean(),
}

monitoring_table = pd.DataFrame(
    [
        {
            "metric": key,
            "current_value": value
        }
        for key, value in monitoring_stats.items()
    ]
)

print(
    monitoring_table.to_string(index=False)
)

                  metric  current_value
                    rows   30000.000000
                 clients      32.000000
            decline_rate       0.542067
             review_rate       0.609867
      median_impressions     731.000000
         median_sessions       7.000000
       median_word_count    2605.000000
median_days_since_update      20.000000
        no_position_rate       0.040167


In [8]:
# ---------------------------------------------------------
# Reason-code monitoring
# ---------------------------------------------------------

reason_counts = (
    queue["reason_codes"]
    .str.split("|")
    .explode()
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="pages")
)

reason_counts["share"] = (
    reason_counts["pages"] / len(queue)
)

print(
    reason_counts.to_string(index=False)
)

print("\nThese counts can be compared with a future run.")

                reason_code  pages    share
                    monitor  11704 0.390133
       low_ctr_visible_page   9759 0.325300
low_engagement_visible_page   7113 0.237100
        page_one_decay_risk   7076 0.235867
          thin_visible_page   5790 0.193000
         stale_visible_page     17 0.000567

These counts can be compared with a future run.


In [9]:
# ---------------------------------------------------------
# Optional performance trigger from ML-09
# ---------------------------------------------------------

if "after_p50" in globals():
    validated_precision_at_50 = after_p50

    trigger_threshold = (
        validated_precision_at_50 * 0.80
    )

    print(
        "Validated Precision@50:",
        round(validated_precision_at_50, 3)
    )

    print(
        "Investigation threshold:",
        round(trigger_threshold, 3)
    )

    print(
        "Trigger rule: investigate if new Precision@50 "
        "falls below this threshold."
    )
else:
    print(
        "ML-09 Precision@50 not found in this runtime."
    )
    print(
        "Record the validated Precision@50 from ML-09 "
        "before using this monitoring trigger."
    )

ML-09 Precision@50 not found in this runtime.
Record the validated Precision@50 from ML-09 before using this monitoring trigger.


## 5. Exports for the paper

I will export the ranked action queue and summary tables to `work/outputs/`.

The queue is the main operational artifact. The summary files provide supporting evidence for the paper, including the reason-code distribution and monitoring statistics.

These files contain anonymized/content-level analytical fields only and are intended to support the documented research workflow.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# Export the ranked queue
# ---------------------------------------------------------

queue_export_columns = [
    "rank",
    "content_id",
    "action_score",
    "action",
    "reason_codes",
    "impressions_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "is_declining_label",
]

queue_export_columns = [
    c for c in queue_export_columns
    if c in queue.columns
]

queue_path = (
    OUTPUT_DIR /
    "content_action_playbook_queue.csv"
)

queue[
    queue_export_columns
].to_csv(
    queue_path,
    index=False
)

print("Saved:", queue_path)

Saved: work/outputs/content_action_playbook_queue.csv


In [11]:
# ---------------------------------------------------------
# Export top-20 review queue
# ---------------------------------------------------------

top20_path = (
    OUTPUT_DIR /
    "content_action_playbook_top20.csv"
)

queue[
    queue_export_columns
].head(20).to_csv(
    top20_path,
    index=False
)

print("Saved:", top20_path)

Saved: work/outputs/content_action_playbook_top20.csv


In [12]:
# ---------------------------------------------------------
# Export reason-code summary
# ---------------------------------------------------------

reason_path = (
    OUTPUT_DIR /
    "content_action_reason_summary.csv"
)

reason_counts.to_csv(
    reason_path,
    index=False
)

print("Saved:", reason_path)

Saved: work/outputs/content_action_reason_summary.csv


In [13]:
# ---------------------------------------------------------
# Export monitoring baseline
# ---------------------------------------------------------

monitoring_path = (
    OUTPUT_DIR /
    "content_action_monitoring_baseline.csv"
)

monitoring_table.to_csv(
    monitoring_path,
    index=False
)

print("Saved:", monitoring_path)

Saved: work/outputs/content_action_monitoring_baseline.csv


In [14]:
# ---------------------------------------------------------
# Final export check
# ---------------------------------------------------------

expected_files = [
    queue_path,
    top20_path,
    reason_path,
    monitoring_path,
]

print("Export check")
print("=" * 40)

for path in expected_files:
    print(
        path,
        "->",
        "OK" if path.exists() else "MISSING"
    )

assert all(path.exists() for path in expected_files)

print("\nAll ML-10 exports created successfully.")

Export check
work/outputs/content_action_playbook_queue.csv -> OK
work/outputs/content_action_playbook_top20.csv -> OK
work/outputs/content_action_reason_summary.csv -> OK
work/outputs/content_action_monitoring_baseline.csv -> OK

All ML-10 exports created successfully.


## ML-10 conclusion

The Content Action Playbook turns the observed model and baseline signals into a ranked review queue with human-readable reason codes.

Its intended role is prioritization: pages with stronger measured signals are reviewed first, while the final content decision remains with a person.

The playbook is not a guarantee of future performance. Its recommendations should be monitored against new measured outcomes, and the model should be investigated or retrained when performance or the underlying data changes materially.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.